In [ ]:
# Standard library
import json
import random
import time
from argparse import ArgumentParser

# Third-party
# for logging the model:
import pytorch_lightning as pl
import torch
from lightning_fabric.utilities import seed
from loguru import logger

# Local
from neural_lam import utils
from neural_lam.config import load_config_and_datastore
from neural_lam.models import EDM, GraphCast, GraphEFM, GraphFM
from neural_lam.weather_dataset import WeatherDataModule

In [ ]:
# Define args class
class Args:
    """Container for all arguments with default values."""
    
    def __init__(self):
        # Basic settings
        self.config_path = "/proj/berzelius-2022-164/users/x_erila/baltic-sea-forecasting/data/baltic_nl_config_small.yaml"
        self.model = "EDM"
        self.seed = 42
        self.num_workers = 4
        self.num_nodes = 1
        self.devices = ["auto"]
        self.epochs = 200
        self.batch_size = 1
        self.load = None
        self.restore_opt = False
        self.precision = 32
        
        # Model architecture
        self.graph = "hierarchical"
        self.hidden_dim = 128
        self.latent_dim = None
        self.hidden_layers = 1
        self.processor_layers = 3
        self.encoder_processor_layers = 1
        self.prior_processor_layers = 1
        self.mesh_aggr = "sum"
        self.output_std = False
        self.prior_dist = "isotropic"
        self.learn_prior = 1
        self.vertical_propnets = 1
        
        # EDM options
        self.sigma_min = 0.002
        self.sigma_max = 88
        self.sigma_data = 1
        self.rho = 7
        self.sampler = "heun"
        self.sampler_steps = 20
        self.diffusion_model = "graph_diff"
        self.pred_residual = True
        
        # Training options
        self.ar_steps_train = 1
        self.loss = "wmse"
        self.lr = 1e-3
        self.val_interval = 1
        self.kl_beta = 1.0
        self.crps_weight = 0
        self.sample_obs_noise = 0
        self.num_sanity_val_steps = 4
        
        # Evaluation options
        self.eval = None
        self.ar_steps_eval = 1
        self.n_example_pred = 1
        self.num_latents_plot = 4
        
        # Logger Settings
        self.logger = "wandb"
        self.logger_project = "baltic-sea-forecasting"
        self.val_steps_to_log = [1, 2, 3]
        self.metrics_watch = []
        self.var_leads_metrics_watch = "{}"
        self.var_leads_val_plot = "{}"
        self.num_past_forcing_steps = 1
        self.num_future_forcing_steps = 1
        self.ensemble_size = 5

# Create the args object with all default values
args = Args()


In [ ]:
# Initialize dataset

# Load neural-lam configuration and datastore to use
config, datastore = load_config_and_datastore(config_path=args.config_path)

# Create datamodule
data_module = WeatherDataModule(
    datastore=datastore,
    ar_steps_train=args.ar_steps_train,
    ar_steps_eval=args.ar_steps_eval,
    standardize=True,
    num_past_forcing_steps=args.num_past_forcing_steps,
    num_future_forcing_steps=args.num_future_forcing_steps,
    batch_size=args.batch_size,
    num_workers=args.num_workers,
)

# Important: Call these methods first to prepare and set up the datasets
data_module.prepare_data()
data_module.setup()

# Get 1 sample
# Get the train dataloader
train_dataloader = data_module.train_dataloader()

# Get a single batch
batch = next(iter(train_dataloader))

init_states, target_states, forcing_features, time = batch

In [ ]:
# Experiment with a trained EDM model
from neural_lam.models import EDM
import torch

# Create the model instance
model = EDM(args, config=config, datastore=datastore)

# Load the weights from checkpoint
checkpoint_path = "/proj/berzelius-2022-164/weather/neural_lam_datasets/baltic/10y-data/last.ckpt"

# Set weights_only=False to allow loading all types of objects
state_dict = torch.load(checkpoint_path, map_location="cpu", weights_only=False)["state_dict"]

# Remove 'model.' prefix if present in keys
if all(k.startswith('model.') for k in state_dict.keys()):
    state_dict = {k[6:]: v for k, v in state_dict.items()}
model.load_state_dict(state_dict, strict=False)

# Move model to available device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# Set model to evaluation mode
model.eval()

print(f"Successfully loaded model from {checkpoint_path}")

In [ ]:
target_state = target_states[:, 0]  # (B, N, F)
prev_prev_state = init_states[:, 0]
prev_state = init_states[:, 1]
forcing = forcing_features[:, 0]  # (B, N_grid, F_forcing)

if args.pred_residual:
    target = target_state - prev_state
else:
    target = (target_state - model.diff_mean) / model.diff_std 


feature_idx = 0  # Choose which weather variable to plot (e.g., temperature)

# Define range for the visualization
vrange = (target[:, feature_idx].min(), target[:, feature_idx].max())

In [ ]:
# Plotting function
from neural_lam.weather_dataset import WeatherDataset
from typing import List, Union
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
from neural_lam.datastore.base import BaseRegularGridDatastore

split = "train"
target_slice = target  # (T, N, F)
time_slice = time[0, :]  # (T,)


def create_dataarray_from_tensor(
    tensor: torch.Tensor,
    time: Union[int, List[int]],
    split: str,
    category: str,
) -> xr.DataArray:
    """
    Create an `xr.DataArray` from a tensor, with the correct dimensions and
    coordinates to match the datastore used by the model. This function in
    in effect is the inverse of what is returned by
    `WeatherDataset.__getitem__`.

    Parameters
    ----------
    tensor : torch.Tensor
        The tensor to convert to a `xr.DataArray` with dimensions [time,
        grid_index, feature]. The tensor will be copied to the CPU if it is
        not already there.
    time : Union[int,List[int]]
        The time index or indices for the data, given as integers or a list
        of integers representing epoch time in nanoseconds. The ints will be
        copied to the CPU memory if they are not already there.
    split : str
        The split of the data, either 'train', 'val', or 'test'
    category : str
        The category of the data, either 'state' or 'forcing'
    """
    # TODO: creating an instance of WeatherDataset here on every call is
    # not how this should be done but whether WeatherDataset should be
    # provided to ARModel or where to put plotting still needs discussion
    weather_dataset = WeatherDataset(datastore=datastore, split=split)
    time = np.array(time.cpu(), dtype="datetime64[ns]")
    da = weather_dataset.create_dataarray_from_tensor(
        tensor=tensor, time=time, category=category
    )
    return da

def plot_prediction(
    datastore: BaseRegularGridDatastore,
    da_prediction: xr.DataArray = None,
    da_target: xr.DataArray = None,
    title=None,
    vrange=None,
):
    """
    Plot example prediction and grond truth.

    Each has shape (N_grid,)

    """
    # Get common scale for values
    if vrange is None:
        vmin = min(da_prediction.min(), da_target.min())
        vmax = max(da_prediction.max(), da_target.max())
    else:
        vmin, vmax = vrange

    extent = datastore.get_xy_extent("state")

    # Set up masking of border region
    da_mask = datastore.unstack_grid_coords(datastore.boundary_mask).isel(
        mask_feature=0
    )
    mask_values = np.invert(da_mask.values.astype(bool)).astype(float)
    pixel_alpha = mask_values.clip(0.7, 1)  # Faded border region

    fig, axes = plt.subplots(
        1,
        2,
        figsize=(21, 7),
        subplot_kw={"projection": datastore.coords_projection},
    )

    # Plot pred and target
    for ax, da in zip(axes, (da_target, da_prediction)):
        ax.coastlines()  # Add coastline outlines
        da.plot.imshow(
            ax=ax,
            origin="lower",
            x="longitude",
            y="latitude",
            extent=extent,
            alpha=pixel_alpha,
            vmin=vmin,
            vmax=vmax,
            cmap="plasma",
            transform=datastore.coords_projection,
        )

    # Ticks and labels
    axes[0].set_title("Ground Truth", size=15)
    axes[1].set_title("Prediction", size=15)

    if title:
        fig.suptitle(title, size=20)

    plt.tight_layout()

    return fig

da_target = create_dataarray_from_tensor(
    tensor=target_slice,
    time=time_slice,
    split=split,
    category="state",
).unstack("grid_index")

feature_idx = 0  # Choose which weather variable to plot (e.g., temperature)
time_idx = 0     # Choose which time step to plot

# Select a single time and feature from the DataArray
da_target_selected = da_target.isel(time=time_idx, state_feature=feature_idx)

fig = plot_prediction(
    datastore=datastore,
    da_prediction=da_target_selected,
    da_target=da_target_selected,
    title="Example Prediction vs Ground Truth",
    vrange=vrange
)


# Save video function

In [ ]:
# Diffusion
sigma_max = 88
sigma_min = 0.002
rho = 7
pred_residual = False
n_steps = 10

rnd_uniform = torch.rand([target.shape[0], 1, 1], device=target.device)
print(f"T: {rnd_uniform}")
rho_inv = 1 / rho
sigma_max_rho = sigma_max**rho_inv
sigma_min_rho = sigma_min**rho_inv
sigma = (
    sigma_max_rho + rnd_uniform * (sigma_min_rho - sigma_max_rho)
) ** rho

y = target

print(f"Sigma: {sigma.squeeze().item():.4f}")
n = torch.randn_like(y) * sigma
noisy_input = y + n

da_noisy = create_dataarray_from_tensor(
    tensor=noisy_input,
    time=time_slice,
    split=split,
    category="state",
).unstack("grid_index")

feature_idx = 0  # Choose which weather variable to plot (e.g., temperature)
time_idx = 0     # Choose which time step to plot

# Select a single time and feature from the DataArray
da_noisy_selected = da_noisy.isel(time=time_idx, state_feature=feature_idx)

fig = plot_prediction(
    datastore=datastore,
    da_prediction=da_noisy_selected,
    da_target=da_target_selected,
    vrange=vrange,
    title=f"Diffusion Ground Truth vs Noisy Sample t = {rnd_uniform.squeeze().item():.2f}",
)

In [ ]:
import os
import matplotlib.pyplot as plt
import imageio

num_steps = 20
sigma_max = 88
sigma_min = 0.002
time_idx = 0     # Choose which time step to plot

# Time step discretization.
step_indices = torch.arange(num_steps)
t_steps = (
    sigma_max ** (1 / rho)
    + step_indices
    / (num_steps - 1)
    * (sigma_min ** (1 / rho) - sigma_max ** (1 / rho))
) ** rho
diff_t_steps = torch.cat(
    [
        torch.as_tensor(t_steps),
        torch.zeros_like(t_steps[:1]),
    ]
)  # t_N = 0

diffusion_steps = []

# Create output directory for frames
frames_dir = "./output/diffusion_frames/mlotst"
os.makedirs(frames_dir, exist_ok=True)
output_path = "./output/diffusion_frames/mlotst.mp4"

frame_files = []

# Loop through sigma values and create frames
for i, sigma in enumerate(diff_t_steps):
    print(f"Processing frame {i+1}/{len(diff_t_steps)}: Sigma = {sigma.item():.4f}")
    n = torch.randn_like(y) * sigma
    noisy_input = y + n
    diffusion_steps.append(noisy_input)

    da_noisy = create_dataarray_from_tensor(
        tensor=noisy_input,
        time=time_slice,
        split=split,
        category="state"
    ).unstack("grid_index")
    
    da_noisy_selected = da_noisy.isel(time=time_idx, state_feature=feature_idx)
    
    fig = plot_prediction(
        datastore=datastore,
        da_prediction=da_noisy_selected,
        da_target=da_target_selected,
        vrange=vrange,
        title=f"Diffusion Process - σ = {sigma.item():.4f}"
    )
    
    # Save this frame
    frame_path = os.path.join(frames_dir, f"frame_{i:03d}.png")
    fig.savefig(frame_path)
    plt.close(fig)
    frame_files.append(frame_path)

# Compile frames into video
print("Creating video...")
with imageio.get_writer(output_path, fps=4) as writer:
    for frame_path in frame_files:
        writer.append_data(imageio.imread(frame_path))

print(f"Video saved to {output_path}")

In [ ]:
# Flow Matching
tmin = 0.0
tmax = 1.0

# Create output directory for frames
frames_dir = "./output/flow_matching/mlotst"
os.makedirs(frames_dir, exist_ok=True)
output_path = "./output/flow_matching/mlotst.mp4"
frame_files = []
fm_steps = []

# Time step discretization.
t_steps = torch.linspace(tmin, tmax, num_steps)

for i, t in enumerate(t_steps):
    z0 = torch.randn_like(target)
    z1 = target
    zt = (1 - t) * z0 + t * z1
    fm_steps.append(zt)

    da_noisy = create_dataarray_from_tensor(
        tensor=zt,
        time=time_slice,
        split=split,
        category="state"
    ).unstack("grid_index")
    
    da_noisy_selected = da_noisy.isel(time=time_idx, state_feature=feature_idx)
    
    fig = plot_prediction(
        datastore=datastore,
        da_prediction=da_noisy_selected,
        da_target=da_target_selected,
        vrange=vrange,
        title=f"t = {t:.4f}"
    )
    
    # Save this frame
    frame_path = os.path.join(frames_dir, f"frame_{i:03d}.png")
    fig.savefig(frame_path)
    plt.close(fig)
    frame_files.append(frame_path)

# Compile frames into video
print("Creating video...")
with imageio.get_writer(output_path, fps=4) as writer:
    for frame_path in frame_files:
        writer.append_data(imageio.imread(frame_path))

print(f"Video saved to {output_path}")
    


In [ ]:
import os
import matplotlib.pyplot as plt
import numpy as np
import imageio
from scipy import stats

# Create output directories for frames and video
dist_frames_dir = "./output/distribution_comparison_with_gt/frames"
os.makedirs(dist_frames_dir, exist_ok=True)
video_path = "./output/distribution_comparison_with_gt/dist_comparison.mp4"
os.makedirs(os.path.dirname(video_path), exist_ok=True)

frame_files = []

# Extract ground truth data (will be the same for all steps)
ground_truth = target[0,:,feature_idx].cpu().numpy()

# Loop through all steps
for step in range(len(fm_steps)):
    print(f"Processing step {step+1}/{len(fm_steps)}")
    
    # Extract the data
    fm_sample = fm_steps[step][0,:,feature_idx].cpu().numpy()
    diff_sample = diffusion_steps[step][0,:,feature_idx].cpu().numpy()

    # Create figure for histogram comparison - SET FIXED FIGURE SIZE
    fig, ax = plt.subplots(1, 3, figsize=(20, 6))
    
    # Plot histograms
    ax[0].hist(ground_truth, bins=50, alpha=0.5, color='black', label='Ground Truth')
    ax[0].hist(fm_sample, bins=50, alpha=0.5, color='blue', label='Flow Matching')
    ax[0].hist(diff_sample, bins=50, alpha=0.5, color='red', label='Diffusion')
    ax[0].set_title('Overlay Histogram Comparison')
    ax[0].set_xlabel('Pixel Value')
    ax[0].set_ylabel('Count')
    ax[0].legend()

    # Plot kernel density estimates for smoother comparison
    gt_kde = stats.gaussian_kde(ground_truth)
    fm_kde = stats.gaussian_kde(fm_sample)
    diff_kde = stats.gaussian_kde(diff_sample)
    
    # Find common x-range
    min_val = min(ground_truth.min(), fm_sample.min(), diff_sample.min())
    max_val = max(ground_truth.max(), fm_sample.max(), diff_sample.max())
    x_range = np.linspace(min_val, max_val, 1000)

    ax[1].plot(x_range, gt_kde(x_range), color='black', label='Ground Truth')
    ax[1].plot(x_range, fm_kde(x_range), color='blue', label='Flow Matching')
    ax[1].plot(x_range, diff_kde(x_range), color='red', label='Diffusion')
    ax[1].set_title('Kernel Density Estimate Comparison')
    ax[1].set_xlabel('Pixel Value')
    ax[1].set_ylabel('Density')
    ax[1].legend()
    
    # Add a text panel with statistics
    ax[2].axis('off')  # No axes for the text panel
    
    # Calculate statistics for all distributions
    gt_stats = {
        'Mean': np.mean(ground_truth),
        'Std Dev': np.std(ground_truth),
        'Min': np.min(ground_truth),
        'Max': np.max(ground_truth),
        'Median': np.median(ground_truth)
    }
    
    fm_stats = {
        'Mean': np.mean(fm_sample),
        'Std Dev': np.std(fm_sample),
        'Min': np.min(fm_sample),
        'Max': np.max(fm_sample)
    }

    diff_stats = {
        'Mean': np.mean(diff_sample),
        'Std Dev': np.std(diff_sample),
        'Min': np.min(diff_sample),
        'Max': np.max(diff_sample)
    }
    
    # Calculate additional comparison metrics
    ks_gt_fm = stats.ks_2samp(ground_truth, fm_sample)
    ks_gt_diff = stats.ks_2samp(ground_truth, diff_sample)
    ks_fm_diff = stats.ks_2samp(fm_sample, diff_sample)
    
    # Create statistics text
    stats_text = f"Step {step}/{len(fm_steps)-1}\n"
    stats_text += f"t_flow = {t_steps[step]:.4f}, σ_diff = {diff_t_steps[step]:.4f}\n\n"
    
    stats_text += "Ground Truth Statistics:\n"
    for stat, value in gt_stats.items():
        stats_text += f"  {stat}: {value:.4f}\n"
    
    stats_text += "\nFlow Matching Statistics:\n"
    for stat, value in fm_stats.items():
        stats_text += f"  {stat}: {value:.4f}\n"
    
    stats_text += "\nDiffusion Statistics:\n"
    for stat, value in diff_stats.items():
        stats_text += f"  {stat}: {value:.4f}\n"
    
    stats_text += "\nKS Tests:\n"
    stats_text += f"  GT vs FM: stat={ks_gt_fm[0]:.4f}, p={ks_gt_fm[1]:.4e}\n"
    stats_text += f"  GT vs Diff: stat={ks_gt_diff[0]:.4f}, p={ks_gt_diff[1]:.4e}\n"
    stats_text += f"  FM vs Diff: stat={ks_fm_diff[0]:.4f}, p={ks_fm_diff[1]:.4e}\n"
    
    # Add the text to the figure
    ax[2].text(0.1, 0.5, stats_text, fontsize=10, va='center', family='monospace')
    
    # Add overall title
    fig.suptitle(f"Distribution Comparison - Step {step}/{len(fm_steps)-1}", fontsize=16)
    
    # Use plt.tight_layout with fixed padding to ensure consistent size
    plt.tight_layout(pad=2.0)
    
    # Save this frame WITH FIXED DPI AND BBOX_INCHES
    frame_path = os.path.join(dist_frames_dir, f"frame_{step:03d}.png")
    fig.savefig(frame_path, dpi=100, bbox_inches='tight', pad_inches=0.5)
    plt.close(fig)
    frame_files.append(frame_path)
    
    # Print statistics to console as well
    print(stats_text)
    print("-" * 50)

# Read first image to get dimensions
first_img = imageio.imread(frame_files[0])
height, width, _ = first_img.shape

# Create video from the frames with consistent size
print("Creating distribution comparison video...")
with imageio.get_writer(video_path, fps=2) as writer:
    for frame_path in frame_files:
        # Read frame and resize to match first frame if needed
        img = imageio.imread(frame_path)
        if img.shape[0] != height or img.shape[1] != width:
            from PIL import Image
            img = np.array(Image.fromarray(img).resize((width, height)))
        writer.append_data(img)

print(f"Video saved to {video_path}")

In [ ]:
import gc

# Clear PyTorch's CUDA cache
torch.cuda.empty_cache()

# Force garbage collection
gc.collect()

# Print memory usage to monitor
print(f"GPU memory allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
print(f"GPU memory reserved: {torch.cuda.memory_reserved() / 1e9:.2f} GB")

In [ ]:
import os

torch.cuda.empty_cache() # Clear GPU memory

# Create output directory for frames
frames_dir = "./output/diffusion_model_steps/mlotst"
os.makedirs(frames_dir, exist_ok=True)
video_path = "./output/diffusion_model_steps/mlotst.mp4"
frame_files = []
diffusion_model_steps = []

def heun_sampler(
        model,
        latents,
        class_labels=None,
        boundary_forcing=None,
        randn_like=torch.randn_like,
        num_steps=20,
        sigma_min=0.03,
        sigma_max=80,
        rho=7,
    ):
        with torch.no_grad():
            # Adjust noise levels based on what's supported by the network.
            sigma_min = max(sigma_min, model.sigma_min)
            sigma_max = min(sigma_max, model.sigma_max)

            # Time step discretization.
            step_indices = torch.arange(num_steps)
            t_steps = (
                sigma_max ** (1 / rho)
                + step_indices
                / (num_steps - 1)
                * (sigma_min ** (1 / rho) - sigma_max ** (1 / rho))
            ) ** rho
            t_steps = torch.cat(
                [
                    torch.as_tensor(t_steps, device=latents.device),
                    torch.zeros_like(t_steps[:1], device=latents.device),
                ]
            )  # t_N = 0

            # Main sampling loop.
            x_next = latents * t_steps[0]

            da_noisy = create_dataarray_from_tensor(
                tensor=x_next.detach(),
                time=time_slice,
                split=split,
                category="state"
            ).unstack("grid_index")
            
            da_noisy_selected = da_noisy.isel(time=time_idx, state_feature=feature_idx)
            
            fig = plot_prediction(
                datastore=datastore,
                da_prediction=da_noisy_selected,
                da_target=da_target_selected,
                vrange=vrange,
                title=f"sigma = {sigma_max:.4f}"
            )

            frame_path = os.path.join(frames_dir, f"frame_{0:03d}.png")
            fig.savefig(frame_path, dpi=100, bbox_inches='tight')
            plt.close(fig)
            frame_files.append(frame_path)

            diffusion_model_steps.append(x_next)
            torch.cuda.empty_cache()  # Add this after each append

            for i, (t_cur, t_next) in enumerate(
                zip(t_steps[:-1], t_steps[1:])
            ):  # 0, ..., N-1
                x_cur = x_next
                print(f"Step {i+1}/{num_steps}: t_cur={t_cur:.4f}, t_next={t_next:.4f}")
                
                # Save this frame
                frame_path = os.path.join(frames_dir, f"frame_{i:03d}.png")
                fig.savefig(frame_path)
                plt.close(fig)
                frame_files.append(frame_path)

                denoised = model.denoise(x_cur, t_cur, class_labels=class_labels)
                d_cur = (x_cur - denoised) / t_cur
                x_next = x_cur + (t_next - t_cur) * d_cur

                # Apply 2nd order correction.
                if i < num_steps - 1:
                    denoised = model.denoise(
                        x_next, t_next, class_labels=class_labels
                    )
                    d_prime = (x_next - denoised) / t_next
                    x_next = x_cur + (t_next - t_cur) * (
                        0.5 * d_cur + 0.5 * d_prime
                    )

                da_noisy = create_dataarray_from_tensor(
                    tensor=x_next.detach(),
                    time=time_slice,
                    split=split,
                    category="state"
                ).unstack("grid_index")
                
                da_noisy_selected = da_noisy.isel(time=time_idx, state_feature=feature_idx)
                
                fig = plot_prediction(
                    datastore=datastore,
                    da_prediction=da_noisy_selected,
                    da_target=da_target_selected,
                    vrange=vrange,
                    title=f"sigma = {t_cur:.4f}"
                )

                frame_path = os.path.join(frames_dir, f"frame_{i+1:03d}.png")
                fig.savefig(frame_path, dpi=100, bbox_inches='tight')
                plt.close(fig)
                frame_files.append(frame_path)

                diffusion_model_steps.append(x_next)
                torch.cuda.empty_cache()  # Add this after each append

            # Read first image to get dimensions
            first_img = imageio.imread(frame_files[0])
            height, width, _ = first_img.shape

            # Create video from the frames with consistent size
            print("Creating distribution comparison video...")
            with imageio.get_writer(video_path, fps=2) as writer:
                for frame_path in frame_files:
                    # Read frame and resize to match first frame if needed
                    img = imageio.imread(frame_path)
                    if img.shape[0] != height or img.shape[1] != width:
                        from PIL import Image
                        img = np.array(Image.fromarray(img).resize((width, height)))
                    writer.append_data(img)

            print(f"Video saved to {video_path}")

            return x_next

latents = torch.randn_like(target).to(device)
input_grid = torch.cat(
    (prev_state, prev_prev_state, forcing), dim=-1
).to(device)  # (B, N_grid, d_input)

pred = heun_sampler(
    model=model,
    latents=latents,
    num_steps=20,
    sigma_min=0.002,
    sigma_max=80,
    rho=7,
    class_labels=input_grid
)



In [ ]:
import os
import matplotlib.pyplot as plt
import numpy as np
import imageio
from scipy import stats
from PIL import Image

# Create output directories for frames and video
dist_frames_dir = "./output/four_way_distribution_comparison/frames"
os.makedirs(dist_frames_dir, exist_ok=True)
video_path = "./output/four_way_distribution_comparison/comparison.mp4"
os.makedirs(os.path.dirname(video_path), exist_ok=True)

frame_files = []

# Extract ground truth data (will be the same for all steps)
ground_truth = target[0,:,feature_idx].cpu().numpy()

# Ensure all lists have the same length for comparison
steps_to_compare = min(len(fm_steps), len(diffusion_steps), len(diffusion_model_steps))

# Loop through all steps
for step in range(steps_to_compare):
    print(f"Processing step {step+1}/{steps_to_compare}")
    
    # Extract the data
    fm_sample = fm_steps[step][0,:,feature_idx].cpu().numpy()
    diff_sample = diffusion_steps[step][0,:,feature_idx].cpu().numpy()
    model_diff_sample = diffusion_model_steps[step][0,:,feature_idx].cpu().numpy()

    # Create figure for comparison - SET FIXED FIGURE SIZE
    fig, ax = plt.subplots(2, 2, figsize=(20, 12))
    
    # Color scheme for consistency
    gt_color = 'black'
    fm_color = 'blue'
    diff_color = 'red'
    model_diff_color = 'green'

    # Plot histograms (all distributions overlaid)
    ax[0,0].hist(ground_truth, bins=50, alpha=0.4, color=gt_color, label='Ground Truth')
    ax[0,0].hist(fm_sample, bins=50, alpha=0.4, color=fm_color, label='Flow Matching')
    ax[0,0].hist(diff_sample, bins=50, alpha=0.4, color=diff_color, label='Diffusion')
    ax[0,0].hist(model_diff_sample, bins=50, alpha=0.4, color=model_diff_color, label='Model Diffusion')
    ax[0,0].set_title('Overlay Histogram Comparison')
    ax[0,0].set_xlabel('Pixel Value')
    ax[0,0].set_ylabel('Count')
    ax[0,0].legend()

    # Plot kernel density estimates
    gt_kde = stats.gaussian_kde(ground_truth)
    fm_kde = stats.gaussian_kde(fm_sample)
    diff_kde = stats.gaussian_kde(diff_sample)
    model_diff_kde = stats.gaussian_kde(model_diff_sample)
    
    # Find common x-range for all distributions
    min_val = min(ground_truth.min(), fm_sample.min(), diff_sample.min(), model_diff_sample.min())
    max_val = max(ground_truth.max(), fm_sample.max(), diff_sample.max(), model_diff_sample.max())
    x_range = np.linspace(min_val, max_val, 1000)

    # Plot density curves
    ax[0,1].plot(x_range, gt_kde(x_range), color=gt_color, label='Ground Truth')
    ax[0,1].plot(x_range, fm_kde(x_range), color=fm_color, label='Flow Matching')
    ax[0,1].plot(x_range, diff_kde(x_range), color=diff_color, label='Diffusion')
    ax[0,1].plot(x_range, model_diff_kde(x_range), color=model_diff_color, label='Model Diffusion')
    ax[0,1].set_title('Kernel Density Estimate Comparison')
    ax[0,1].set_xlabel('Pixel Value')
    ax[0,1].set_ylabel('Density')
    ax[0,1].legend()
    
    # Add box plots for quick visual comparison
    box_data = [ground_truth, fm_sample, diff_sample, model_diff_sample]
    ax[1,0].boxplot(box_data, labels=['Ground Truth', 'Flow Matching', 'Diffusion', 'Model Diffusion'])
    ax[1,0].set_title('Box Plot Comparison')
    ax[1,0].set_ylabel('Pixel Value')
    
    # Add text panel with statistics
    ax[1,1].axis('off')  # No axes for the text panel
    
    # Calculate statistics for all distributions
    gt_stats = {
        'Mean': np.mean(ground_truth),
        'Std Dev': np.std(ground_truth),
        'Min': np.min(ground_truth),
        'Max': np.max(ground_truth),
        'Median': np.median(ground_truth)
    }
    
    fm_stats = {
        'Mean': np.mean(fm_sample),
        'Std Dev': np.std(fm_sample),
        'Min': np.min(fm_sample),
        'Max': np.max(fm_sample),
        'Median': np.median(fm_sample)
    }

    diff_stats = {
        'Mean': np.mean(diff_sample),
        'Std Dev': np.std(diff_sample),
        'Min': np.min(diff_sample),
        'Max': np.max(diff_sample),
        'Median': np.median(diff_sample)
    }
    
    model_diff_stats = {
        'Mean': np.mean(model_diff_sample),
        'Std Dev': np.std(model_diff_sample),
        'Min': np.min(model_diff_sample),
        'Max': np.max(model_diff_sample),
        'Median': np.median(model_diff_sample)
    }
    
    # Calculate statistical tests between ground truth and each method
    ks_gt_fm = stats.ks_2samp(ground_truth, fm_sample)
    ks_gt_diff = stats.ks_2samp(ground_truth, diff_sample)
    ks_gt_model = stats.ks_2samp(ground_truth, model_diff_sample)
    
    # Create statistics text
    stats_text = f"Step {step}/{steps_to_compare-1}\n"
    stats_text += f"t_flow = {t_steps[step]:.4f}, σ_diff = {diff_t_steps[step]:.4f}\n\n"
    
    stats_text += "Ground Truth Statistics:\n"
    for stat, value in gt_stats.items():
        stats_text += f"  {stat}: {value:.4f}\n"
    
    stats_text += "\nFlow Matching Statistics:\n"
    for stat, value in fm_stats.items():
        stats_text += f"  {stat}: {value:.4f}\n"
    
    stats_text += "\nDiffusion Statistics:\n"
    for stat, value in diff_stats.items():
        stats_text += f"  {stat}: {value:.4f}\n"
    
    stats_text += "\nModel Diffusion Statistics:\n"
    for stat, value in model_diff_stats.items():
        stats_text += f"  {stat}: {value:.4f}\n"
    
    stats_text += "\nKS Tests vs Ground Truth:\n"
    stats_text += f"  GT vs FM: stat={ks_gt_fm[0]:.4f}, p={ks_gt_fm[1]:.4e}\n"
    stats_text += f"  GT vs Diff: stat={ks_gt_diff[0]:.4f}, p={ks_gt_diff[1]:.4e}\n"
    stats_text += f"  GT vs Model: stat={ks_gt_model[0]:.4f}, p={ks_gt_model[1]:.4e}\n"
    
    # Add the text to the figure
    ax[1,1].text(0.0, 0.5, stats_text, fontsize=9, va='center', family='monospace')
    
    # Add overall title
    fig.suptitle(f"Distribution Comparison - Step {step}/{steps_to_compare-1}", fontsize=16)
    
    # Use tight_layout with fixed parameters
    plt.tight_layout(pad=2.0)
    
    # Save this frame - USE FIXED DPI and PAD INCHES to ensure consistent size
    frame_path = os.path.join(dist_frames_dir, f"frame_{step:03d}.png")
    fig.savefig(frame_path, dpi=100, bbox_inches=None)
    plt.close(fig)
    frame_files.append(frame_path)
    
    # Print statistics to console as well
    print(stats_text)
    print("-" * 50)

# Create video from the frames with consistent size
print("Creating distribution comparison video...")

# Two approaches to handle this:

# OPTION 1: Read all images first and resize to the same dimensions
print("Reading images and standardizing dimensions...")
frames = []
for frame_path in frame_files:
    img = Image.open(frame_path)
    frames.append(np.array(img))

# Get the modal height and width (most common dimensions)
heights = [frame.shape[0] for frame in frames]
widths = [frame.shape[1] for frame in frames]
from collections import Counter
height = Counter(heights).most_common(1)[0][0]  
width = Counter(widths).most_common(1)[0][0]

# Resize all frames to the modal dimensions
print(f"Standardizing all frames to {width}x{height}")
standardized_frames = []
for frame in frames:
    if frame.shape[0] != height or frame.shape[1] != width:
        resized_img = np.array(Image.fromarray(frame).resize((width, height)))
        standardized_frames.append(resized_img)
    else:
        standardized_frames.append(frame)

# Create video
with imageio.get_writer(video_path, fps=2) as writer:
    for frame in standardized_frames:
        writer.append_data(frame)

print(f"Video saved to {video_path}")

In [ ]:
diff_t_steps

In [ ]:
diff_t_steps[1:] - diff_t_steps[:-1]